# Segmentacion de clientes

In [ ]:
# Instalación manual si falta una dependencia: %pip install kneed -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import plotly as py
import plotly.graph_objs as go

# Para el escalado
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from kneed import KneeLocator

from sklearn.cluster import KMeans

El dataset Mall_Customers.csv contiene información de clientes de un centro comercial. Las columnas incluyen:

- CustomerID
- Gender
- Age
- Annual Income (k$)
- Spending Score (1-100)

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
df.head()

In [ ]:
plt.figure(1 , figsize = (15 , 6))
n = 0
for x in ['Age' , 'Annual Income (k$)' , 'Spending Score (1-100)']:
    n += 1
    plt.subplot(1 , 3 , n)
    plt.subplots_adjust(hspace = 0.5 , wspace = 0.5)
    sns.histplot(df[x] , bins = 15, kde=True)
    plt.title('Distplot of {}'.format(x))
plt.show()

In [ ]:
sns.pairplot(df, vars = ['Spending Score (1-100)', 'Annual Income (k$)', 'Age'], hue = "Gender")

# Selección de características

In [ ]:
features = ['Age' , 'Annual Income (k$)' ,'Spending Score (1-100)']
X = df[features].copy()
X.head()

# Escalado

In [ ]:
# para tener los features bajo la misma escala de valores
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
kmeans_kwargs = {
   "init": "random",
   "n_init": 10,
   "max_iter": 300,
   "random_state": 42,
   }

# Método del codo

In [ ]:
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit(X)
    sse.append(kmeans.inertia_)

In [ ]:
plt.plot(range(1, 11), sse)
plt.xticks(range(1, 11))
plt.xlabel("Número de Clusters")
plt.ylabel("SSE")

plt.show()

In [ ]:
kl = KneeLocator(
    range(1, 11), sse, curve="convex", direction="decreasing"
    )

In [ ]:
kl.elbow

# Método de la silueta

In [ ]:
silhouette_coefficients = []

   # Notice you start at 2 clusters for silhouette coefficient
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit(X)
    score = silhouette_score(X, kmeans.labels_)
    silhouette_coefficients.append(score)

In [ ]:
plt.plot(range(2, 11), silhouette_coefficients)
plt.xticks(range(2, 11))
plt.xlabel("Numero de Clusters")
plt.ylabel("Coeficiente Silhouette")
plt.show()

In [ ]:
# Crear el modelo de clusterización
kmeans = KMeans(n_clusters=2 + silhouette_coefficients.index(max(silhouette_coefficients)), **kmeans_kwargs)

# Entrenamiendo y predicción del grupo
# Creación de columna 'cluster' para tener el número de grupo
df["Cluster"] = kmeans.fit_predict(X)

# Definir la columna 'cluster' como tipo de dato categórico
df["Cluster"] = df["Cluster"].astype("category")

# Visualización

In [ ]:
import plotly as py
import plotly.graph_objs as go

trace1 = go.Scatter3d(
    x= df['Age'],
    y= df['Spending Score (1-100)'],
    z= df['Annual Income (k$)'],
    mode='markers',
     marker=dict(
        color = df['Cluster'],
        size= 10,
        line=dict(
            color= df['Cluster'],
            width= 12
        ),
        opacity=0.8
     )
)
data = [trace1]
layout = go.Layout(
    title= 'Clusters wrt Age, Income and Spending Scores',
    scene = dict(
            xaxis = dict(title  = 'Age'),
            yaxis = dict(title  = 'Spending Score'),
            zaxis = dict(title  = 'Annual Income')
        )
)



fig = go.Figure(data=data, layout=layout)
fig.update_layout(width=1500, height=1000)
py.offline.iplot(fig)
